# Pattern Mining & Visualization for GitHub Technology Transactions

## Project Goal

In this notebook, we will analyze technology combinations from repository transactions.

Each row represents one repository/basket of technologies, for example:

`lang:python | topic:machine-learning | topic:pytorch | topic:tensorflow`

## Required Tasks

We will:

1. Load and inspect the transaction dataset.
2. Clean and transform transactions into basket format.
3. Apply **Apriori** algorithm.
4. Apply **FP-Growth** algorithm.
5. Find frequent technology combinations.
6. Generate association rules.
7. Measure:
   - Support
   - Confidence
   - Lift
   - Conviction
   - Leverage
   - Certainty Factor (CF)
8. Create visualizations for:
   - Programming language trends
   - Frequent technologies
   - Frequent technology combinations
   - Repository/technology co-occurrence network
   - Top repositories by ranking
9. Export final tables and charts.

# Cell 1 — Install Required Libraries

We need several Python libraries:

- `pandas`: data loading and cleaning
- `mlxtend`: Apriori, FP-Growth, and association rules
- `matplotlib`: charts
- `networkx`: graph/network visualization
- `tqdm`: progress display

Run this cell once in Colab.

In [ ]:
!pip -q install mlxtend networkx tqdm

# Cell 2 — Import Libraries

This cell imports all libraries that will be used in the notebook.

In [ ]:
import os
import json
import math
import warnings
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

# Cell 3 — Load the Dataset

## Important

Your uploaded dataset has one main column:

- `transaction`: contains technologies separated by `|`

Example:

`lang:python|topic:machine-learning|topic:pytorch`

Upload `transactions.csv` to Colab files panel, or put it in Google Drive and update `DATA_PATH`.

In [ ]:
# Option A: If you upload transactions.csv directly to Colab
DATA_PATH = "/content/transactions.csv"

# Option B: If you use Google Drive, uncomment and edit this path
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/transactions.csv"

# Fallback: if CSV is not available but JSON is uploaded
JSON_PATH = "/content/transactions.json"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
elif os.path.exists(JSON_PATH):
    df = pd.read_json(JSON_PATH)
else:
    raise FileNotFoundError("Please upload transactions.csv or transactions.json to Colab.")

print("Dataset shape:", df.shape)
df.head()

# Cell 4 — Basic Data Understanding

Before mining patterns, we should understand:

- Number of rows
- Column names
- Missing values
- Example transactions

This is important because association mining depends heavily on clean transaction baskets.

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum())

print("\nSample transactions:")
for i, value in enumerate(df["transaction"].dropna().head(5), start=1):
    print(f"{i}. {value}")

# Cell 5 — Parse Transactions into Lists

Association rule mining needs data in this form:

```python
[
  ['lang:python', 'topic:machine-learning', 'topic:pytorch'],
  ['lang:html', 'topic:chatgpt', 'topic:llm']
]
```

So we split each transaction by `|`, remove spaces, remove empty values, and remove duplicates inside the same transaction.

In [ ]:
def parse_transaction(x):
    if pd.isna(x):
        return []
    items = [item.strip().lower() for item in str(x).split("|")]
    items = [item for item in items if item]
    # remove duplicates while preserving order
    return list(dict.fromkeys(items))

transactions = df["transaction"].apply(parse_transaction).tolist()
transactions = [t for t in transactions if len(t) > 0]

print("Number of valid transactions:", len(transactions))
print("Example parsed transaction:")
transactions[0]

# Cell 6 — Separate Languages and Topics

In the dataset, technologies usually start with prefixes:

- `lang:` means programming language
- `topic:` means repository topic/technology

This helps us visualize programming language trends separately from technology topics.

In [ ]:
all_items = [item for trans in transactions for item in trans]

language_items = [item for item in all_items if item.startswith("lang:")]
topic_items = [item for item in all_items if item.startswith("topic:")]

language_counts = Counter(language_items)
topic_counts = Counter(topic_items)

print("Number of unique items:", len(set(all_items)))
print("Number of unique languages:", len(language_counts))
print("Number of unique topics:", len(topic_counts))

print("\nTop languages:")
print(language_counts.most_common(10))

print("\nTop topics:")
print(topic_counts.most_common(10))

# Cell 7 — Visualize Programming Language Trends

This chart shows the most frequent programming languages in the repositories.

Interpretation:

- A high bar means this language appears in many repository transactions.
- This tells us the dominant languages in the dataset.

In [ ]:
def plot_counter(counter, title, xlabel, ylabel, top_n=15):
    data = pd.DataFrame(counter.most_common(top_n), columns=[xlabel, ylabel])
    plt.figure(figsize=(12, 6))
    plt.barh(data[xlabel][::-1], data[ylabel][::-1])
    plt.title(title)
    plt.xlabel(ylabel)
    plt.ylabel(xlabel)
    plt.tight_layout()
    plt.show()
    return data

lang_trends_df = plot_counter(
    language_counts,
    title="Top Programming Languages",
    xlabel="Language",
    ylabel="Frequency",
    top_n=15
)

lang_trends_df

# Cell 8 — Visualize Frequent Technologies / Topics

This chart shows the most frequent repository topics/technologies.

Examples:

- `topic:machine-learning`
- `topic:deep-learning`
- `topic:pytorch`
- `topic:tensorflow`

This answers: **What technologies are trending most in the dataset?**

In [ ]:
topic_trends_df = plot_counter(
    topic_counts,
    title="Top Repository Topics / Technologies",
    xlabel="Technology / Topic",
    ylabel="Frequency",
    top_n=25
)

topic_trends_df

# Cell 9 — One-Hot Encoding for Pattern Mining

Apriori and FP-Growth cannot work directly on raw text lists.

We need to convert the baskets into one-hot format:

| lang:python | topic:ml | topic:pytorch |
|---|---|---|
| True | True | False |
| False | True | True |

Each column is an item, and each row is a transaction.

In [ ]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print("Basket matrix shape:", basket_df.shape)
basket_df.head()

# Cell 10 — Choose Minimum Support

## What is Minimum Support?

Minimum support controls how frequent an itemset must be to appear in the results.

For example, if:

`min_support = 0.03`

That means the technology combination must appear in at least **3% of repositories**.

Because technology datasets can have many unique topics, we start with a moderate value. If results are too few, lower it. If results are too many, increase it.

In [ ]:
MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.30

print("Minimum support:", MIN_SUPPORT)
print("Minimum confidence:", MIN_CONFIDENCE)
print("Minimum transactions required:", math.ceil(MIN_SUPPORT * len(transactions)))

# Cell 11 — Apply Apriori Algorithm

## Apriori Idea

Apriori finds frequent itemsets using this logic:

> If an itemset is frequent, then all its smaller subsets must also be frequent.

Example:

If `{python, pytorch, deep-learning}` is frequent, then:

- `{python}` must be frequent
- `{pytorch}` must be frequent
- `{deep-learning}` must be frequent
- `{python, pytorch}` must be frequent

This reduces the search space.

In [ ]:
apriori_itemsets = apriori(
    basket_df,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

apriori_itemsets["length"] = apriori_itemsets["itemsets"].apply(len)
apriori_itemsets = apriori_itemsets.sort_values(
    ["length", "support"], ascending=[False, False]
).reset_index(drop=True)

print("Number of frequent itemsets from Apriori:", len(apriori_itemsets))
apriori_itemsets.head(20)

# Cell 12 — Apply FP-Growth Algorithm

## FP-Growth Idea

FP-Growth finds frequent itemsets without generating many candidate itemsets like Apriori.

It builds a compressed structure called an **FP-Tree**.

In practice:

- Apriori is easier to understand.
- FP-Growth is usually faster for larger datasets.

We apply both because the task requires both algorithms.

In [ ]:
fpgrowth_itemsets = fpgrowth(
    basket_df,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

fpgrowth_itemsets["length"] = fpgrowth_itemsets["itemsets"].apply(len)
fpgrowth_itemsets = fpgrowth_itemsets.sort_values(
    ["length", "support"], ascending=[False, False]
).reset_index(drop=True)

print("Number of frequent itemsets from FP-Growth:", len(fpgrowth_itemsets))
fpgrowth_itemsets.head(20)

# Cell 13 — Compare Apriori and FP-Growth Results

Both algorithms should usually produce the same frequent itemsets if they use the same minimum support.

Here we compare how many itemsets each algorithm found.

In [ ]:
comparison = pd.DataFrame({
    "Algorithm": ["Apriori", "FP-Growth"],
    "Number of Frequent Itemsets": [len(apriori_itemsets), len(fpgrowth_itemsets)]
})

comparison

# Cell 14 — Frequent Technology Combinations

Now we focus on itemsets with length >= 2 because the task asks for **technology combinations**.

A combination like:

`{topic:machine-learning, topic:pytorch}`

means these two technologies frequently appear together in repositories.

In [ ]:
frequent_combinations = fpgrowth_itemsets[fpgrowth_itemsets["length"] >= 2].copy()
frequent_combinations = frequent_combinations.sort_values("support", ascending=False)

print("Number of frequent combinations:", len(frequent_combinations))
frequent_combinations.head(30)

# Cell 15 — Visualize Frequent Technology Combinations

This chart shows the most frequent technology combinations.

Interpretation:

- Higher support means the combination appears in more repositories.
- These combinations represent common technology stacks.

In [ ]:
def itemset_to_text(itemset):
    return " + ".join(sorted(list(itemset)))

combo_plot_df = frequent_combinations.head(20).copy()
combo_plot_df["combination"] = combo_plot_df["itemsets"].apply(itemset_to_text)

plt.figure(figsize=(14, 8))
plt.barh(combo_plot_df["combination"][::-1], combo_plot_df["support"][::-1])
plt.title("Top Frequent Technology Combinations")
plt.xlabel("Support")
plt.ylabel("Technology Combination")
plt.tight_layout()
plt.show()

combo_plot_df[["combination", "support", "length"]].head(20)

# Cell 16 — Association Rules: Concept Explanation

Association rules have this form:

`A → B`

Meaning:

> If a repository contains technology A, it is likely to also contain technology B.

Example:

`topic:pytorch → topic:deep-learning`

This means repositories containing `pytorch` often also contain `deep-learning`.

## Metrics

### 1. Support

Support measures how common the full rule is in the whole dataset.

$$
Support(A \rightarrow B) = Support(A \cup B)
$$

Meaning: percentage of repositories containing both A and B.

---

### 2. Confidence

Confidence measures how often B appears when A appears.

$$
Confidence(A \rightarrow B) = \frac{Support(A \cup B)}{Support(A)}
$$

Meaning: among repositories that contain A, how many also contain B?

---

### 3. Lift

Lift tells us whether A and B appear together more than expected by chance.

$$
Lift(A \rightarrow B) = \frac{Confidence(A \rightarrow B)}{Support(B)}
$$

Interpretation:

- `Lift > 1`: positive relationship
- `Lift = 1`: independent/no special relationship
- `Lift < 1`: negative relationship

---

### 4. Conviction

Conviction measures how strongly A implies B by considering how often the rule would be wrong.

$$
Conviction(A \rightarrow B) = \frac{1 - Support(B)}{1 - Confidence(A \rightarrow B)}
$$

Interpretation:

- Higher conviction means stronger implication.
- If confidence = 1, conviction becomes infinity because the rule has no observed violations.

---

### 5. Leverage

Leverage measures the difference between actual co-occurrence and expected co-occurrence if A and B were independent.

$$
Leverage(A \rightarrow B) = Support(A \cup B) - Support(A) \times Support(B)
$$

Interpretation:

- Positive leverage: A and B appear together more than expected.
- Zero leverage: independent.
- Negative leverage: appear together less than expected.

---

### 6. Certainty Factor (CF)

Certainty Factor compares confidence with the original probability of B.

If confidence is greater than support(B):

$$
CF = \frac{Confidence - Support(B)}{1 - Support(B)}
$$

If confidence is less than support(B):

$$
CF = \frac{Confidence - Support(B)}{Support(B)}
$$

Interpretation:

- Positive CF: A increases belief in B.
- Zero CF: A gives no extra information about B.
- Negative CF: A decreases belief in B.

# Cell 17 — Generate Association Rules

We use frequent itemsets from FP-Growth, then generate rules using minimum confidence.

We also add:

- Antecedent length
- Consequent length
- Certainty Factor
- Clean text columns for easier reading

In [ ]:
def certainty_factor(confidence, consequent_support):
    if pd.isna(confidence) or pd.isna(consequent_support):
        return np.nan
    if consequent_support == 0:
        return np.nan
    if confidence > consequent_support:
        return (confidence - consequent_support) / (1 - consequent_support)
    elif confidence < consequent_support:
        return (confidence - consequent_support) / consequent_support
    else:
        return 0.0

rules = association_rules(
    fpgrowth_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

rules["antecedent_len"] = rules["antecedents"].apply(len)
rules["consequent_len"] = rules["consequents"].apply(len)
rules["CF"] = rules.apply(lambda row: certainty_factor(row["confidence"], row["consequent support"]), axis=1)

rules["antecedents_text"] = rules["antecedents"].apply(itemset_to_text)
rules["consequents_text"] = rules["consequents"].apply(itemset_to_text)
rules["rule"] = rules["antecedents_text"] + "  →  " + rules["consequents_text"]

selected_cols = [
    "rule", "antecedents_text", "consequents_text",
    "antecedent support", "consequent support", "support",
    "confidence", "lift", "conviction", "leverage", "CF",
    "antecedent_len", "consequent_len"
]

rules_clean = rules[selected_cols].sort_values(
    ["lift", "confidence", "support"], ascending=[False, False, False]
).reset_index(drop=True)

print("Number of association rules:", len(rules_clean))
rules_clean.head(30)

# Cell 18 — Explain Each Top Association Rule Automatically

This cell generates a readable interpretation for each rule.

For every rule, we explain:

- What the rule means
- Support meaning
- Confidence meaning
- Lift meaning
- Conviction meaning
- Leverage meaning
- CF meaning

This is very useful for your report or presentation.

In [ ]:
def explain_rule(row):
    rule = row["rule"]
    ant = row["antecedents_text"]
    con = row["consequents_text"]
    support = row["support"]
    confidence = row["confidence"]
    lift_value = row["lift"]
    conviction = row["conviction"]
    leverage = row["leverage"]
    cf = row["CF"]
    cons_support = row["consequent support"]
    ant_support = row["antecedent support"]

    if lift_value > 1:
        lift_msg = "positive relationship: they appear together more than expected by chance"
    elif lift_value == 1:
        lift_msg = "almost independent relationship"
    else:
        lift_msg = "negative relationship: they appear together less than expected by chance"

    if cf > 0:
        cf_msg = "the antecedent increases our belief in the consequent"
    elif cf < 0:
        cf_msg = "the antecedent decreases our belief in the consequent"
    else:
        cf_msg = "the antecedent gives almost no extra belief about the consequent"

    return f"""
Rule: {rule}
- Meaning: repositories containing [{ant}] tend to also contain [{con}].
- Support = {support:.4f}: {support*100:.2f}% of all repositories contain both sides of the rule.
- Antecedent support = {ant_support:.4f}: {ant_support*100:.2f}% of repositories contain the left side.
- Consequent support = {cons_support:.4f}: {cons_support*100:.2f}% of repositories contain the right side.
- Confidence = {confidence:.4f}: among repositories containing [{ant}], {confidence*100:.2f}% also contain [{con}].
- Lift = {lift_value:.4f}: this indicates a {lift_msg}.
- Conviction = {conviction:.4f}: higher values mean fewer rule violations.
- Leverage = {leverage:.4f}: positive values mean actual co-occurrence is higher than random expectation.
- Certainty Factor = {cf:.4f}: {cf_msg}.
"""

TOP_N_RULES_TO_EXPLAIN = 10
for i, (_, row) in enumerate(rules_clean.head(TOP_N_RULES_TO_EXPLAIN).iterrows(), start=1):
    print("="*100)
    print(f"Explanation {i}")
    print(explain_rule(row))

# Cell 19 — Visualize Top Association Rules by Lift

Lift is useful because it highlights strong relationships, not just frequent ones.

A rule with high confidence but low lift may be obvious because the consequent is already very common.

High lift means the relationship is more interesting.

In [ ]:
top_lift_rules = rules_clean.head(20).copy()

plt.figure(figsize=(14, 8))
plt.barh(top_lift_rules["rule"][::-1], top_lift_rules["lift"][::-1])
plt.title("Top Association Rules by Lift")
plt.xlabel("Lift")
plt.ylabel("Rule")
plt.tight_layout()
plt.show()

top_lift_rules[["rule", "support", "confidence", "lift", "conviction", "leverage", "CF"]]

# Cell 20 — Scatter Plot: Support vs Confidence

This visualization helps us understand rule quality.

- X-axis: Support
- Y-axis: Confidence
- Bubble size: Lift

Good rules usually have reasonable support, high confidence, and lift greater than 1.

In [ ]:
plot_rules = rules_clean.copy()
plot_rules = plot_rules.replace([np.inf, -np.inf], np.nan).dropna(subset=["support", "confidence", "lift"])

plt.figure(figsize=(10, 7))
plt.scatter(
    plot_rules["support"],
    plot_rules["confidence"],
    s=np.clip(plot_rules["lift"], 1, 10) * 30,
    alpha=0.6
)
plt.title("Association Rules: Support vs Confidence")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.tight_layout()
plt.show()

# Cell 21 — Repository / Technology Co-occurrence Network

A network graph shows technologies as nodes.

An edge between two technologies means they appear together in repositories.

The stronger the co-occurrence, the more important the connection.

To keep the graph readable, we only use top frequent technologies and top edges.

In [ ]:
TOP_ITEMS_FOR_NETWORK = 30
TOP_EDGES_FOR_NETWORK = 80

# choose top items overall
item_counts = Counter(all_items)
top_items = set([item for item, count in item_counts.most_common(TOP_ITEMS_FOR_NETWORK)])

edge_counter = Counter()
for trans in transactions:
    filtered = sorted([item for item in trans if item in top_items])
    for a, b in combinations(filtered, 2):
        edge_counter[(a, b)] += 1

G = nx.Graph()
for item in top_items:
    G.add_node(item, frequency=item_counts[item])

for (a, b), weight in edge_counter.most_common(TOP_EDGES_FOR_NETWORK):
    G.add_edge(a, b, weight=weight)

plt.figure(figsize=(16, 12))
pos = nx.spring_layout(G, k=0.6, seed=42)
node_sizes = [G.nodes[n]["frequency"] * 8 for n in G.nodes]
edge_widths = [G[u][v]["weight"] / 20 for u, v in G.edges]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, alpha=0.8)
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.4)
nx.draw_networkx_labels(G, pos, font_size=9)

plt.title("Technology Co-occurrence Network")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

# Cell 22 — Top Repositories by Ranking

Your uploaded file contains only a `transaction` column, so there is no original repository name/stars/ranking column.

To still satisfy the visualization requirement, we create a simple analytical ranking:

## Repository Score

We rank each row/repository using:

1. Number of technologies in the repository.
2. Sum of global frequencies of those technologies.

Meaning:

A repository gets a higher score if it contains many common/trending technologies.

If your original dataset later includes columns like `repo_name`, `stars`, `forks`, or `rank`, we can replace this score with the real ranking.

In [ ]:
repo_df = pd.DataFrame({
    "repo_id": [f"repo_{i+1}" for i in range(len(transactions))],
    "items": transactions
})

repo_df["num_technologies"] = repo_df["items"].apply(len)
repo_df["frequency_score"] = repo_df["items"].apply(lambda items: sum(item_counts[item] for item in items))
repo_df["ranking_score"] = repo_df["num_technologies"] * 0.4 + repo_df["frequency_score"] * 0.6
repo_df["technologies"] = repo_df["items"].apply(lambda x: " | ".join(x))

repo_ranking = repo_df.sort_values("ranking_score", ascending=False).reset_index(drop=True)
repo_ranking.head(20)[["repo_id", "num_technologies", "frequency_score", "ranking_score", "technologies"]]

# Cell 23 — Visualize Top Repositories by Ranking

This chart shows top repositories based on the analytical ranking score.

Again, this is a fallback because the uploaded dataset does not contain real repository names or stars.

In [ ]:
top_repos = repo_ranking.head(15)

plt.figure(figsize=(12, 6))
plt.barh(top_repos["repo_id"][::-1], top_repos["ranking_score"][::-1])
plt.title("Top Repositories by Analytical Ranking Score")
plt.xlabel("Ranking Score")
plt.ylabel("Repository")
plt.tight_layout()
plt.show()

top_repos[["repo_id", "num_technologies", "frequency_score", "ranking_score"]]

# Cell 24 — Export Outputs

The final required outputs are:

- Frequent itemsets
- Association rules
- Charts and graph

This cell saves the main tables as CSV files so you can submit them or include them in your report.

In [ ]:
os.makedirs("outputs", exist_ok=True)

apriori_itemsets_export = apriori_itemsets.copy()
apriori_itemsets_export["itemsets"] = apriori_itemsets_export["itemsets"].apply(itemset_to_text)

fpgrowth_itemsets_export = fpgrowth_itemsets.copy()
fpgrowth_itemsets_export["itemsets"] = fpgrowth_itemsets_export["itemsets"].apply(itemset_to_text)

frequent_combinations_export = frequent_combinations.copy()
frequent_combinations_export["itemsets"] = frequent_combinations_export["itemsets"].apply(itemset_to_text)

apriori_itemsets_export.to_csv("outputs/apriori_frequent_itemsets.csv", index=False)
fpgrowth_itemsets_export.to_csv("outputs/fpgrowth_frequent_itemsets.csv", index=False)
frequent_combinations_export.to_csv("outputs/frequent_technology_combinations.csv", index=False)
rules_clean.to_csv("outputs/association_rules_with_metrics.csv", index=False)
repo_ranking.to_csv("outputs/repository_ranking.csv", index=False)
lang_trends_df.to_csv("outputs/programming_language_trends.csv", index=False)
topic_trends_df.to_csv("outputs/frequent_technologies.csv", index=False)

print("Files saved inside outputs/ folder:")
for file in os.listdir("outputs"):
    print("-", file)

# Cell 25 — Final Summary

## What We Did

We completed the full **Pattern Mining & Visualization** pipeline:

1. Loaded transaction data.
2. Parsed technologies from each repository transaction.
3. Visualized programming language trends.
4. Visualized frequent technologies/topics.
5. Converted transactions to one-hot encoding.
6. Applied Apriori.
7. Applied FP-Growth.
8. Extracted frequent technology combinations.
9. Generated association rules.
10. Calculated support, confidence, lift, conviction, leverage, and CF.
11. Explained top rules automatically.
12. Built a technology co-occurrence network.
13. Created a fallback repository ranking visualization.
14. Exported final CSV outputs.

## Key Interpretation Guide

- **High support**: common pattern.
- **High confidence**: strong implication from left side to right side.
- **Lift > 1**: meaningful positive association.
- **High conviction**: rule has fewer violations.
- **Positive leverage**: technologies appear together more than random expectation.
- **Positive CF**: the left side increases belief in the right side.